# Nightlife API Data

This notebook follows the process of gathering and transforming collected nightlife data for each destination. Data was gathered from Apify's Google Maps Extractor API. Particular focus will be placed on data transformation, converting the data into a clean, unified, and standardised format. 

## Import Python Libraries

In [3]:
#data handling 
import pandas as pd 
import numpy as np 

#file and path management 
from pathlib import Path as path 

## Load and Concat Each Dataset

In [14]:
#define path to raw nightlife data
nightlife_raw = path("data")/"raw_data"/"nightlife" #create a path object 

#create a list of destinations to loop through 
destinations = ['ibiza','mykonos','tenerife','bali','bangkok','cancun',
                'dubrovnik','holetown','maldives','miami','phuket','sal']

#empty list for files to be added to 
destination_files = []

#for loop to locate each individual file and concat to a df 
for d in destinations: 
    
    #list of files to loop through 
    csv_file = nightlife_raw/f"{d}_nightlife_data.csv"
    #convert each file to a df
    df = pd.read_csv(csv_file) 
    df['destination'] = d.title() #add column with destination name to join with other destination dfs  
    destination_files.append(df) #add each df to files list 

nightlife_data = pd.concat(destination_files, ignore_index=True) 
print(f"Total rows loaded: {len(nightlife_data)}")

Total rows loaded: 348


## EDA

In [6]:
nightlife_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 348 entries, 0 to 347
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   imageUrl      339 non-null    object 
 1   title         348 non-null    object 
 2   totalScore    339 non-null    float64
 3   reviewsCount  339 non-null    float64
 4   street        326 non-null    object 
 5   city          340 non-null    object 
 6   state         228 non-null    object 
 7   countryCode   348 non-null    object 
 8   website       177 non-null    object 
 9   phone         249 non-null    object 
 10  categoryName  348 non-null    object 
 11  url           348 non-null    object 
dtypes: float64(2), object(10)
memory usage: 32.8+ KB


In [7]:
nightlife_data.isna().sum()

imageUrl          9
title             0
totalScore        9
reviewsCount      9
street           22
city              8
state           120
countryCode       0
website         171
phone            99
categoryName      0
url               0
dtype: int64

In [ ]:
nightlife_data["countryCode"].unique()

## Transform Data

1. Create a `'country'` column using `'countryCode'`
2. Drop unwanted columns
3. Drop records with null `'totalScore'` and `'reviewsCount'` values
4. Sort `'categoryName'` into 3 categories: restaurant, bar, and club
5. Aggregate and export nightlife data

Null records will be dropped as opposed to being filled, as filling values creates non-existent data and will produce misleading results.

### 1. Create country column

In [19]:
#dictionary of country code and country names 
country_codes_names = {
    'ES': 'Spain', 
    'GR': 'Greece',
    'ID': 'Indonesia',
    'TH': 'Thailand',
    'MX': 'Mexico', 
    'HR': 'Croatia',
    'BB': 'Barbados',
    'MV': 'Maldives',
    'US': 'USA', 
     'CV': 'Cape Verde'
}

#replace countryCode column values with country name
nightlife_data["country"] = nightlife_data["countryCode"].replace(country_codes_names)

### 2. Drop unwanted columns

In [21]:
#list of columns to drop 
cols_to_drop = ['imageUrl','street','city','state','countryCode','website','phone','url']

#drop unwanted cols 
nightlife_data = nightlife_data.drop(columns=cols_to_drop)

### 3. Drop null records

In [24]:
nightlife_data = nightlife_data.dropna()

### 4. Categorise nightlife type

In [26]:
#identify subcategories 
nightlife_data["categoryName"].unique()

array(['Disco club', 'Cocktail bar', 'Bar', 'Lounge bar', 'Night club',
       'Bar & grill', 'Restaurant', 'Karaoke bar', 'Stand bar',
       'Seafood restaurant', 'Gay bar', 'Tapas bar', 'Tavern', 'Pub',
       'Sports bar', 'Night market', 'Luggage storage facility',
       'Live music bar', 'Beer hall', 'Wine bar', 'Tiki bar',
       'Wine store', 'Cafe', 'Piano bar', 'Fine dining restaurant',
       'Club', 'Mediterranean restaurant', 'Tea house', 'Beach club'],
      dtype=object)

In [37]:
#dictionary of categories and subcategories  

nightlife_cat = {
    'Food': ['Restaurant', 'Seafood restaurant', 'Cafe', 'Fine dining restaurant', 'Mediterranean restaurant'],
    'Bar': ['Cocktail bar','Bar', 'Lounge bar', 'Bar & grill', 'Karaoke bar', 'Stand bar', 'Gay bar', 'Tapas bar', 'Tavern',
           'Pub', 'Sports bar', 'Live music bar', 'Beer hall', 'Wine bar', 'Tiki bar', 'Piano bar'], 
    'Club': ['Disco club', 'Night club', 'Club', 'Beach club'], 
    'Other': ['Luggage storage facility', 'Wine store', 'Night market', 'Tea house']
}

#create a reverse dict to replace subcat values with category key 
nightlife_cat_rev = {
    subcat: category #subcat values become keys, cat becomes values 
    #search for each key,value pair and find each subcat in the list 
    for category, values in nightlife_cat.items()  
    for subcat in values
}

#invert dictionary and replaces values with keys 
nightlife_data["category"] = nightlife_data["categoryName"].replace(nightlife_cat_rev)

#drop categoryName column 

nightlife_data = nightlife_data.drop(columns='categoryName')

/var/folders/kq/bffmzf714b35b4nwcqx3925c0000gn/T/ipykernel_94650/3029207333.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nightlife_data["category"] = nightlife_data["categoryName"].replace(nightlife_cat_rev)


### Export non-aggregate nightlife data

A non-aggregated dataframe will be exported for app users to explore the different nightlife options in each destination.

In [40]:
#export to csv file
nightlife_data.to_csv('destinations_nightlife_data', index=False)

### 5. Aggregate and export nightlife data

In [73]:
#group by cols 
group_by_cols = ['destination','country']

#aggregation dictionary assigning calculation to column 
agg_dict = {
    'averageRating': ('totalScore','mean'), 
    'totalReviews': ('reviewsCount','sum'), 
}

nightlife_data_agg = nightlife_data.groupby(group_by_cols).agg(
    **agg_dict).reset_index()

In [74]:
#round numerical columns 

nightlife_data_agg['averageRating'] = round(nightlife_data_agg['averageRating'],2)
nightlife_data_agg['totalReviews'] = round(nightlife_data_agg['totalReviews'])

In [75]:
nightlife_data_agg.to_csv('destinations_nightlife_agg.csv', index=False)